# 07 - Severity Modeling

**Purpose:** Select the predictors for the claim severity model in each of the
four coverages, and fit the final GLM. Candidate variables are added by forward
stepwise selection, scored on out-of-sample deviance against the holdout set.
The distribution family is taken from 05; this notebook decides the mean
structure only.

**Input:** `data/processed/claims.csv`
**Output:** Fitted severity GLMs and their relativities, carried into
08_RatingAlgorithm.
**Depends on:** 02_Cleaned_Data, 05_Severity_Distribution

**Specification:** Gamma GLM with an explicit log link. Note that statsmodels
defaults Gamma to the inverse link, which would break the multiplicative rating
structure, so the link must be passed explicitly.

**Candidate predictors:** same list as 06. Fitted on the claims table, so each
row is one claim and the model is conditional on a claim having occurred.

**Constraint:** claim counts are 973 / 528 / 233 / 85 by coverage, so
credibility is the binding limitation here rather than in 06.

**Author:** Dalton | **Last run:** 2026-09-27


Setup

In [10]:
from pathlib import Path
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf

ROOT = Path.cwd().parent
PROC = ROOT / "data" / "processed"

pd.set_option('display.max_colwidth', None)

claims = pd.read_csv(PROC / "claims.csv")          # 1,819 individual claims

# 'class' is a Python keyword, so patsy formulas cannot use it
claims = claims.rename(columns={'class': 'class_year'})

COVERAGES = ['Personal Property', 'Additional Living Expense',
             'Guest Medical', 'Liability']

# training only; holdout is sealed until validation
sv_train = {c: claims[(claims.coverage == c) &
                      (claims.training_holdout == 'training')].reset_index(drop=True)
            for c in COVERAGES}

sv_hold  = {c: claims[(claims.coverage == c) &
                      (claims.training_holdout == 'holdout')].reset_index(drop=True)
            for c in COVERAGES}

RATING_VARS = ['class_year', 'study', 'gpa', 'greek', 'gender',
               'off_campus', 'distance_to_campus', 'sprinklered']

pd.DataFrame({
    'train_claims': {c: len(sv_train[c]) for c in COVERAGES},
    'hold_claims':  {c: len(sv_hold[c])  for c in COVERAGES},
    'mean_severity': {c: sv_train[c].amount.mean().round(0) for c in COVERAGES},
})

CAT_VARS = ['class_year', 'study', 'greek', 'gender', 'off_campus']
CATS = {c: pd.Categorical(claims[c]).categories for c in CAT_VARS}

Create function to get holdout deviance and select parameters

In [11]:
def lognormal_deviance(y, mu_log):
    """Gaussian deviance on the log scale: sum of squared residuals of log(y)."""
    y      = np.asarray(y, dtype=float)
    mu_log = np.asarray(mu_log, dtype=float)
    return float(np.sum((np.log(y) - mu_log) ** 2))


def forward_stepwise_lognormal(train, hold, candidates):
    """Forward selection on out-of-sample log-scale deviance.
    Log-normal is fitted as a Gaussian GLM on log(amount) with identity link.
    Continues while any addition improves holdout deviance.
    Returns (selected variables, one combined table of every step)."""
    keep     = []
    m_cur    = smf.glm('np.log(amount) ~ 1', data=train,
                       family=sm.families.Gaussian()).fit()
    d_cur    = lognormal_deviance(hold.amount, m_cur.predict(hold))
    all_rows = []
    step     = 0

    while True:
        remaining = [v for v in candidates if v not in keep]
        if not remaining:
            break
        step += 1

        rows = []
        if step == 1:
            rows.append({'step': step, 'model': '(null)',
                         'train_AIC': m_cur.aic, 'holdout_deviance': d_cur})

        results = {}
        for v in remaining:
            f = 'np.log(amount) ~ ' + ' + '.join(keep + [v])
            m = smf.glm(f, data=train, family=sm.families.Gaussian()).fit()
            d = lognormal_deviance(hold.amount, m.predict(hold))
            results[v] = (m, d)
            rows.append({'step': step,
                         'model': ' + '.join(keep + [v]),
                         'train_AIC': m.aic, 'holdout_deviance': d})

        all_rows.extend(rows)

        best = min(results, key=lambda v: results[v][1])

        if results[best][1] >= d_cur:
            break

        keep.append(best)
        m_cur, d_cur = results[best]

    return keep, (pd.DataFrame(all_rows)
                    .sort_values('holdout_deviance')
                    .reset_index(drop=True))

Now to run the function on Personal Property

In [12]:
pp_sv = sv_train['Personal Property'][RATING_VARS + ['amount']].copy()
for c in CAT_VARS:
    pp_sv[c] = pd.Categorical(pp_sv[c], categories=CATS[c])

pp_sv_hold = sv_hold['Personal Property'][RATING_VARS + ['amount']].copy()
for c in CAT_VARS:
    pp_sv_hold[c] = pd.Categorical(pp_sv_hold[c], categories=CATS[c])

In [16]:
pp_sv_keep, pp_sv_path = forward_stepwise_lognormal(pp_sv, pp_sv_hold, RATING_VARS)
pp_sv_path.round(3)

,step,model,train_AIC,holdout_deviance
0,5,greek + off_campus + sprinklered + study + distance_to_campus,1626.741,67.301
1,4,greek + off_campus + sprinklered + study,1624.822,67.317
2,4,greek + off_campus + sprinklered + distance_to_campus,1624.188,67.494
3,3,greek + off_campus + sprinklered,1622.362,67.510
4,6,greek + off_campus + sprinklered + study + distance_to_campus + gpa,1628.547,67.585
5,5,greek + off_campus + sprinklered + study + gpa,1626.621,67.605
6,6,greek + off_campus + sprinklered + study + distance_to_campus + gender,1629.976,67.729
7,5,greek + off_campus + sprinklered + study + gender,1628.061,67.758
8,4,greek + off_campus + sprinklered + gpa,1624.106,67.859
9,4,greek + off_campus + sprinklered + gender,1625.465,68.044


Based on above, we will use greek, off-campus, sprinkelered, study, and distance to campus. Note that the VIF will not change with a new repsonse variable to we will continue using the collinear predictors

Now Getting predictors for additional living expense

In [14]:
ale_sv = sv_train['Additional Living Expense'][RATING_VARS + ['amount']].copy()
for c in CAT_VARS:
    ale_sv[c] = pd.Categorical(ale_sv[c], categories=CATS[c])

ale_sv_hold = sv_hold['Additional Living Expense'][RATING_VARS + ['amount']].copy()
for c in CAT_VARS:
    ale_sv_hold[c] = pd.Categorical(ale_sv_hold[c], categories=CATS[c])

In [15]:
ale_sv_keep, ale_sv_path = forward_stepwise_lognormal(ale_sv, ale_sv_hold, RATING_VARS)
ale_sv_path.round(3)

,step,model,train_AIC,holdout_deviance
0,3,greek + off_campus + class_year,887.012,36.803
1,4,greek + off_campus + class_year + distance_to_campus,889.001,36.815
2,4,greek + off_campus + class_year + gpa,888.944,36.902
3,4,greek + off_campus + class_year + study,888.003,37.272
4,4,greek + off_campus + class_year + sprinklered,826.153,37.445
5,4,greek + off_campus + class_year + gender,883.427,39.353
6,3,greek + off_campus + distance_to_campus,898.323,40.939
7,3,greek + off_campus + gpa,898.336,40.976
8,2,greek + off_campus,896.336,40.976
9,3,greek + off_campus + sprinklered,843.576,41.428


Based on above, for additional living expense severity we will use the predictors greek, off campus, and class year

Now for Guest Medical predictions

In [17]:
gm_sv = sv_train['Guest Medical'][RATING_VARS + ['amount']].copy()
for c in CAT_VARS:
    gm_sv[c] = pd.Categorical(gm_sv[c], categories=CATS[c])

gm_sv_hold = sv_hold['Guest Medical'][RATING_VARS + ['amount']].copy()
for c in CAT_VARS:
    gm_sv_hold[c] = pd.Categorical(gm_sv_hold[c], categories=CATS[c])

In [18]:
gm_sv_keep, gm_sv_path = forward_stepwise_lognormal(gm_sv, gm_sv_hold, RATING_VARS)
gm_sv_path.round(3)

,step,model,train_AIC,holdout_deviance
0,2,greek + class_year,541.572,41.626
1,3,greek + class_year + gpa,542.695,41.831
2,3,greek + class_year + sprinklered,526.053,42.058
3,3,greek + class_year + distance_to_campus,538.415,42.661
4,2,greek + sprinklered,528.773,43.277
5,3,greek + class_year + off_campus,523.000,44.228
6,3,greek + class_year + gender,545.513,44.334
7,1,greek,540.563,44.378
8,2,greek + gpa,542.008,44.383
9,3,greek + class_year + study,541.713,45.371


Thus we will use Greek and class year as our predictors for Guest medical

Now to analyze liability, since it comes from an exponential distribution we need to define a new function with the appropriate linkage function and fit


In [19]:
def gamma_deviance(y, mu):
    """Gamma/exponential deviance: 2 * sum[ -log(y/mu) + (y - mu)/mu ]."""
    y  = np.asarray(y,  dtype=float)
    mu = np.asarray(mu, dtype=float)
    return float(2.0 * np.sum(-np.log(y / mu) + (y - mu) / mu))


def forward_stepwise_exponential(train, hold, candidates):
    """Forward selection on out-of-sample deviance for an exponential GLM.
    Exponential is Gamma with the dispersion fixed at 1, log link.
    Continues while any addition improves holdout deviance.
    Returns (selected variables, one combined table of every step)."""
    fam = sm.families.Gamma(link=sm.families.links.Log())

    keep     = []
    m_cur    = smf.glm('amount ~ 1', data=train, family=fam).fit(scale=1.0)
    d_cur    = gamma_deviance(hold.amount, m_cur.predict(hold))
    all_rows = []
    step     = 0

    while True:
        remaining = [v for v in candidates if v not in keep]
        if not remaining:
            break
        step += 1

        rows = []
        if step == 1:
            rows.append({'step': step, 'model': '(null)',
                         'train_AIC': m_cur.aic, 'holdout_deviance': d_cur})

        results = {}
        for v in remaining:
            f = 'amount ~ ' + ' + '.join(keep + [v])
            m = smf.glm(f, data=train, family=fam).fit(scale=1.0)
            d = gamma_deviance(hold.amount, m.predict(hold))
            results[v] = (m, d)
            rows.append({'step': step,
                         'model': ' + '.join(keep + [v]),
                         'train_AIC': m.aic, 'holdout_deviance': d})

        all_rows.extend(rows)

        best = min(results, key=lambda v: results[v][1])

        if results[best][1] >= d_cur:
            break

        keep.append(best)
        m_cur, d_cur = results[best]

    return keep, (pd.DataFrame(all_rows)
                    .sort_values('holdout_deviance')
                    .reset_index(drop=True))

In [21]:
lia_sv = sv_train['Liability'][RATING_VARS + ['amount']].copy()
for c in CAT_VARS:
    lia_sv[c] = pd.Categorical(lia_sv[c], categories=CATS[c])

lia_sv_hold = sv_hold['Liability'][RATING_VARS + ['amount']].copy()
for c in CAT_VARS:
    lia_sv_hold[c] = pd.Categorical(lia_sv_hold[c], categories=CATS[c])

print(len(lia_sv), len(lia_sv_hold))

67 18


In [22]:
lia_sv_keep, lia_sv_path = forward_stepwise_exponential(lia_sv, lia_sv_hold, RATING_VARS)
lia_sv_path.round(3)

,step,model,train_AIC,holdout_deviance
0,1,distance_to_campus,1343.898,6.911
1,2,distance_to_campus + gpa,1342.995,7.207
2,2,distance_to_campus + greek,1339.875,8.021
3,2,distance_to_campus + sprinklered,1342.874,8.104
4,2,distance_to_campus + study,1348.599,8.331
5,1,off_campus,1342.218,8.493
6,1,gpa,1341.461,8.790
7,2,distance_to_campus + off_campus,1344.208,8.842
8,1,greek,1340.792,8.905
9,1,(null),1342.788,9.417


Based on the above, distance to campus is the only predictor worth using for liability claim severity(Note that holdout sample is low so may need a different approach)

Conclusion: (coefficents table)

In [24]:
import re

def clean_term(t):
    m = re.match(r'^(.*?)\[T\.(.*)\]$', t)
    return f"{m.group(1)} = {m.group(2)}" if m else t

SEV_SPECS = {
    'Personal Property':         {'vars': ['greek', 'off_campus', 'sprinklered',
                                           'study', 'distance_to_campus'],
                                  'fam': 'lognormal'},
    'Additional Living Expense': {'vars': ['greek', 'off_campus', 'class_year'],
                                  'fam': 'lognormal'},
    'Guest Medical':             {'vars': ['greek', 'class_year'],
                                  'fam': 'lognormal'},
    'Liability':                 {'vars': ['distance_to_campus'],
                                  'fam': 'exponential'},
}

CAT_VARS = ['class_year', 'study', 'greek', 'gender', 'off_campus']
CATS     = {c: pd.Categorical(claims[c]).categories for c in CAT_VARS}

rows, SEV_FINAL, SIGMA2 = [], {}, {}

for c, sp in SEV_SPECS.items():
    d = sv_train[c][RATING_VARS + ['amount']].copy()
    for k in CAT_VARS:
        d[k] = pd.Categorical(d[k], categories=CATS[k])

    r = ' + '.join(sp['vars']) if sp['vars'] else '1'

    if sp['fam'] == 'lognormal':
        d = d.assign(log_amount=np.log(d.amount))
        m = smf.glm('log_amount ~ ' + r, data=d,
                    family=sm.families.Gaussian()).fit()
        SIGMA2[c] = m.scale                     # residual variance on the log scale
    else:
        m = smf.glm('amount ~ ' + r, data=d,
                    family=sm.families.Gamma(link=sm.families.links.Log())
                    ).fit(scale=1.0)
        SIGMA2[c] = 0.0                         # no retransformation correction

    SEV_FINAL[c] = m

    for t in m.params.index:
        # fold sigma^2/2 into the intercept so exp(intercept) is baseline E[X]
        b = m.params[t] + (SIGMA2[c] / 2 if t == 'Intercept' else 0.0)
        rows.append({'coverage': c, 'term': clean_term(t),
                     'coef': b, 'pval': m.pvalues[t], 'se': m.bse[t],
                     'component': 'severity'})

sev_coefs = pd.DataFrame(rows)[['coverage', 'term', 'coef', 'pval', 'se', 'component']]
sev_coefs.to_csv(PROC / "sev_coefficients.csv", index=False)

sev_coefs.round(4)

,coverage,term,coef,pval,se,component
0,Personal Property,Intercept,7.8644,0.0000,0.0839,severity
1,Personal Property,greek = Non-greek,-0.6719,0.0000,0.0507,severity
2,Personal Property,off_campus = On campus,-0.4345,0.0000,0.0626,severity
3,Personal Property,sprinklered = True,-0.3788,0.0000,0.0535,severity
4,Personal Property,study = Humanities,-0.0943,0.1675,0.0683,severity
5,Personal Property,study = Other,-0.0827,0.2163,0.0668,severity
6,Personal Property,study = Science,0.0024,0.9715,0.0669,severity
7,Personal Property,distance_to_campus,0.0027,0.7770,0.0095,severity
8,Additional Living Expense,Intercept,9.1577,0.0000,0.0907,severity
9,Additional Living Expense,greek = Non-greek,-0.7577,0.0000,0.0696,severity
